# Part 2 — Tasks a & b: Perceptron from scratch

A single-file perceptron implementation (**Task a**) and its evaluation on
two datasets:

- **Task a** — `SeaSyn` (described as linearly separable), features `attrib1`–`attrib3`.
- **Task b** — `RingSyn` (non-linearly separable, concentric rings), features `feature1`, `feature2`.

Metric: **test-set accuracy** measured after training for each of
`[1, 5, 10, 15, 20, 50, 60, 80, 100, 120, 150, 200]` epochs.

In [1]:
import numpy as np
import pandas as pd

## Step 1 — The perceptron

A perceptron is a single linear unit. For an input `x` it computes the net
input `w · x + b` and fires (predicts class **1**) when that value is
non-negative, otherwise class **0** (a step / threshold activation).

Training uses the classic **perceptron learning rule**: for every sample we
compare the prediction to the target and nudge the weights by
`lr × (target − prediction) × x`. The error term is `+1`, `0`, or `−1`, so
weights only change on misclassified points. Weights start at zero and we
iterate through the training samples in order each epoch, which keeps the
result fully deterministic and reproducible.

In [2]:
class Perceptron:
    """Single-layer perceptron trained with the perceptron learning rule."""

    def __init__(self, n_features, lr=0.01):
        self.w = np.zeros(n_features)   # one weight per feature
        self.b = 0.0                    # bias
        self.lr = lr                    # learning rate

    def net_input(self, x):
        return x @ self.w + self.b

    def predict(self, X):
        # step activation: fire (1) when the net input is non-negative
        return (self.net_input(X) >= 0.0).astype(int)

    def fit(self, X, y, epochs):
        for _ in range(epochs):
            for xi, target in zip(X, y):
                pred = 1 if self.net_input(xi) >= 0.0 else 0
                error = target - pred           # -1, 0, or +1
                self.w += self.lr * error * xi
                self.b += self.lr * error
        return self


def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))

## Step 2 — Load data and sweep over epoch counts

`run_sweep` trains a **fresh** perceptron for each epoch count in the list and
records its accuracy on the test set, returning a tidy results table.

In [3]:
EPOCHS = [1, 5, 10, 15, 20, 50, 60, 80, 100, 120, 150, 200]


def load_xy(path):
    """Read a CSV and split into feature matrix X and integer labels y."""
    df = pd.read_csv(path)
    y = df["class"].to_numpy().astype(int)
    X = df.drop(columns="class").to_numpy(dtype=float)
    return X, y


def run_sweep(train_path, test_path, epochs_list=EPOCHS, lr=0.01):
    """Train a fresh perceptron per epoch count; record test accuracy."""
    X_train, y_train = load_xy(train_path)
    X_test, y_test = load_xy(test_path)
    rows = []
    for n_epochs in epochs_list:
        model = Perceptron(X_train.shape[1], lr=lr)
        model.fit(X_train, y_train, n_epochs)
        acc = accuracy(y_test, model.predict(X_test))
        rows.append({"Model (epoch)": f"Perceptron ({n_epochs})",
                     "Accuracy": round(acc, 4)})
    return pd.DataFrame(rows).set_index("Model (epoch)")

## Task a — SeaSyn (linearly separable data)

Train on `SeaSynTrain.csv`, evaluate on `SeaSynTest.csv`.

In [4]:
sea_results = run_sweep("data/SeaSynTrain.csv", "data/SeaSynTest.csv")
sea_results

,Accuracy
Model (epoch),
Perceptron (1),0.625
Perceptron (5),0.700
Perceptron (10),0.600
Perceptron (15),0.850
Perceptron (20),0.625
Perceptron (50),0.650
Perceptron (60),0.600
Perceptron (80),0.625
Perceptron (100),0.625


### Task a — discussion

**How does increasing the number of epochs affect the model's accuracy?**

Test accuracy **fluctuates within roughly 0.6–0.85** as epochs increase, with
**no steady upward trend** — it peaks early (0.85 at 15 epochs) but sits around
0.6–0.75 at most other epoch counts, including the largest ones. More epochs do
**not** give reliably better accuracy here.

The reason is that the perceptron convergence theorem only guarantees a
zero-error solution when the data is *perfectly* linearly separable. The
provided SeaSyn classes actually overlap — even strong linear models
(logistic regression, linear SVM) top out around ~0.85 — so no separating
hyperplane exists. Because some points are always misclassified, the learning
rule keeps applying non-zero updates every epoch, so the weight vector
**oscillates around the best linear boundary** instead of settling. That is
why the accuracy jumps around with epoch count rather than converging.

## Task b — RingSyn (non-linearly separable data)

Reusing the **exact same** `Perceptron` and `run_sweep` on the ring dataset:
train on `RingSynTrain.csv`, evaluate on `RingSynTest.csv`.

In [5]:
ring_results = run_sweep("data/RingSynTrain.csv", "data/RingSynTest.csv")
ring_results

,Accuracy
Model (epoch),
Perceptron (1),0.6567
Perceptron (5),0.6567
Perceptron (10),0.7200
Perceptron (15),0.6567
Perceptron (20),0.6567
Perceptron (50),0.7500
Perceptron (60),0.6567
Perceptron (80),0.6567
Perceptron (100),0.6567


In [6]:
# For context: the majority-class baseline on the ring test set.
_, y_ring_test = load_xy("data/RingSynTest.csv")
majority = np.bincount(y_ring_test).max() / len(y_ring_test)
print(f"Majority-class baseline (ring test): {majority:.4f}")

Majority-class baseline (ring test): 0.6567


### Task b — discussion

**How well does the perceptron perform on this dataset?**

Poorly. Accuracy stays pinned near **0.66 for almost every epoch count** — the
**majority-class baseline** (0.6567, printed above) — with only minor blips
(e.g. ~0.72–0.75). The perceptron effectively gives up and predicts the larger
class for (nearly) every point, and more epochs do not change this.

This is the expected limitation of a linear classifier. RingSyn consists of
two **concentric rings**: the inner class is surrounded by the outer class, so
no straight line (hyperplane) can separate them. A perceptron can only learn a
linear decision boundary, so the best it can do is fall back to the dominant
class. Adding epochs cannot help, because the problem is the *shape* of the
boundary the model is capable of drawing — not how long it trains. This
motivates Task c, where a multi-layer perceptron adds the non-linearity needed
to carve out a curved boundary.